# GENIE Task 3 — Diffusion Models

Two experiments:
- **latent_diffusion** (main): frozen VAE encoder/decoder + MLP latent denoiser
- **image_ddpm** (baseline): pixel-space DDPM with U-Net

All logic lives in `src/task3_diffusion.py`. This notebook is a thin wrapper.

In [ ]:
# 1. Drive mount
from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_MOUNT_POINT = Path('/content/drive')

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT_POINT))
        print(f'Mounted Google Drive at {DRIVE_MOUNT_POINT}')
    except ImportError:
        print('google.colab is not available in this environment.')
else:
    print('Skipping Google Drive mount.')

In [ ]:
# 2. Clone repo and install dependencies
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nitik1998/GENIE_DiffusionLearning.git'
REPO_DIR = Path('/content/GENIE_DiffusionLearning')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Repo ready at: {REPO_DIR}')
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# 3. Configuration
from pathlib import Path
import json, os, random, shutil, subprocess, sys
import numpy as np, torch
from IPython.display import Image, Markdown, display

SEED = 42

# ═══════════════════════════════════════════════════
# MODE SELECTOR
# ═══════════════════════════════════════════════════
RUN_MODE     = 'sanity'           # 'sanity' (quick) or 'full'
DIFFUSION_MODE = 'latent_diffusion'  # 'latent_diffusion' (main) or 'image_ddpm' (baseline)

# Auto-config based on mode
if RUN_MODE == 'sanity':
    MAX_EVENTS = 1000
    EPOCHS     = 5
    N_SAMPLES  = 6
else:
    MAX_EVENTS = None
    EPOCHS     = 30
    N_SAMPLES  = 8

EXP_NAME   = DIFFUSION_MODE
BATCH_SIZE = 0
TIMESTEPS  = 200

# VAE checkpoint (optional — if you have a trained Task 1 model)
VAE_CHECKPOINT = None  # e.g. '/content/results_colab/checkpoints/vae_best.pt'
# ═══════════════════════════════════════════════════

DATASET_SOURCE = Path('/content/drive/MyDrive/quark-gluon_data-set_n139306.hdf5')
if not DATASET_SOURCE.exists():
    DATASET_SOURCE = Path('/content/quark-gluon_data-set_n139306.hdf5')

RESULTS_ROOT = Path('/content/results_colab')
CHECKPOINT_ROOT = RESULTS_ROOT / 'checkpoints'
DATA_DIR = REPO_DIR / 'data'
DATASET_TARGET = DATA_DIR / 'quark-gluon_data-set_n139306.hdf5'
FINAL_RESULTS_DIR = RESULTS_ROOT / 'task3' / EXP_NAME

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

for p in [DATA_DIR, RESULTS_ROOT, CHECKPOINT_ROOT, FINAL_RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if DATASET_SOURCE.exists() and DATASET_SOURCE.resolve() != DATASET_TARGET.resolve():
    shutil.copy2(DATASET_SOURCE, DATASET_TARGET)
elif not DATASET_TARGET.exists():
    raise FileNotFoundError(f'Dataset not found at {DATASET_SOURCE} or {DATASET_TARGET}')

os.environ['GENIE_PROJECT_ROOT']  = str(REPO_DIR)
os.environ['GENIE_DATA_DIR']      = str(DATA_DIR)
os.environ['GENIE_OUTPUT_DIR']    = str(RESULTS_ROOT)
os.environ['GENIE_CHECKPOINT_DIR']= str(CHECKPOINT_ROOT)

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print(f'Mode: {DIFFUSION_MODE} ({RUN_MODE})')
print(f'Epochs: {EPOCHS}, Max events: {MAX_EVENTS}, Batch: {BATCH_SIZE}')

In [ ]:
# 4. Run Task 3
cmd = [
    sys.executable, '-u',
    'src/task3_diffusion.py',
    '--mode', DIFFUSION_MODE,
    '--exp-name', EXP_NAME,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--timesteps', str(TIMESTEPS),
    '--n-samples', str(N_SAMPLES),
    '--seed', str(SEED),
    '--force-rerun',
]
if MAX_EVENTS is not None:
    cmd += ['--max-events', str(MAX_EVENTS)]
if VAE_CHECKPOINT is not None:
    cmd += ['--vae-checkpoint', VAE_CHECKPOINT]

print('Running:', ' '.join(map(str, cmd)), flush=True)
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Task 3 failed with return code {proc.returncode}')
print('\n✅ Task 3 complete!')

In [ ]:
# 5. Display outputs
def show_png(path, width=900):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing: {path}')

display(Markdown(f'## Results: {EXP_NAME}'))

show_png(FINAL_RESULTS_DIR / 'original_vs_reconstructed.png', width=1200)
show_png(FINAL_RESULTS_DIR / 'original_vs_reconstructed_vs_abs_error.png', width=1400)
show_png(FINAL_RESULTS_DIR / 'training_curves.png', width=900)

# Metrics
metrics_path = FINAL_RESULTS_DIR / 'metrics.json'
if metrics_path.exists():
    payload = json.loads(metrics_path.read_text())
    metrics = payload.get('metrics', payload)
    display(Markdown(f'### Metrics'))
    for k, v in metrics.items():
        print(f'  {k}: {v}')

In [ ]:
# 6. Package artifacts
archive_base = RESULTS_ROOT / EXP_NAME
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(FINAL_RESULTS_DIR.parent), base_dir=FINAL_RESULTS_DIR.name)
print(f'Created archive: {archive_path}')

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print(f'Automatic download skipped: {exc}')